In [3]:

import pandas as pd

print("=" * 80)
print("STEP-BY-STEP AUDIT: SHP → X28 FIRM MATCHING PIPELINE")
print("=" * 80)

# ============================================================
# LOAD ALL DATA
# ============================================================

scrape = pd.read_csv('/Users/bradyallardice/Dropbox/kurer_allardice_technology/data/created/scrape_uid_complete.csv')
uid_final = pd.read_csv('/Users/bradyallardice/Dropbox/kurer_allardice_technology/data/created/shp_uid_final.csv')
x28 = pd.read_csv('/Users/bradyallardice/Dropbox/kurer_allardice_technology/data/created/shp_x28_match.csv')
firmid = pd.read_csv('/Users/bradyallardice/Dropbox/kurer_allardice_technology/data/created/shp_firmid_anon.csv')

print("\n--- DATA FILES LOADED ---")
print(f"  scrape_uid_complete.csv:  {len(scrape):,} rows (unique firm name/location pairs from BUR/MIS)")
print(f"  shp_uid_final.csv:        {len(uid_final):,} rows (cleaned, deduplicated UIDs)")
print(f"  shp_x28_match.csv:        {len(x28):,} rows (X28 firms)")
print(f"  shp_firmid_anon.csv:      {len(firmid):,} rows (person-year records)")


STEP-BY-STEP AUDIT: SHP → X28 FIRM MATCHING PIPELINE

--- DATA FILES LOADED ---
  scrape_uid_complete.csv:  13,643 rows (unique firm name/location pairs from BUR/MIS)
  shp_uid_final.csv:        7,014 rows (cleaned, deduplicated UIDs)
  shp_x28_match.csv:        7,201 rows (X28 firms)
  shp_firmid_anon.csv:      70,111 rows (person-year records)


In [4]:

# ============================================================
# STEP 1: What is scrape_uid_complete.csv?
# ============================================================

print("\n" + "=" * 80)
print("STEP 1: THE SCRAPE FILE (scrape_uid_complete.csv)")
print("=" * 80)
print("""
This file contains ~13.6k unique (firm_name, location) pairs extracted from
the BUR/MIS data. Each was sent to the Swiss UID website. The scraper returned
the official firm name, UID, and match accuracy.
""")

n_total_scrape = len(scrape)
n_got_uid = scrape['bur_uid'].notna().sum()
n_no_uid = scrape['bur_uid'].isna().sum()
n_unique_uids = scrape['bur_uid'].dropna().nunique()

print(f"  Total entries:              {n_total_scrape:,}")
print(f"  Got a UID from scraper:     {n_got_uid:,} ({100*n_got_uid/n_total_scrape:.1f}%)")
print(f"  No UID found:               {n_no_uid:,} ({100*n_no_uid/n_total_scrape:.1f}%)")
print(f"  Unique UIDs returned:       {n_unique_uids:,}")
print(f"  (many names map to same UID = same firm, different name variant)")


STEP 1: THE SCRAPE FILE (scrape_uid_complete.csv)

This file contains ~13.6k unique (firm_name, location) pairs extracted from
the BUR/MIS data. Each was sent to the Swiss UID website. The scraper returned
the official firm name, UID, and match accuracy.

  Total entries:              13,643
  Got a UID from scraper:     11,044 (80.9%)
  No UID found:               2,599 (19.1%)
  Unique UIDs returned:       7,721
  (many names map to same UID = same firm, different name variant)


In [5]:

# ============================================================
# STEP 2: What is shp_uid_final.csv?
# ============================================================

print("\n" + "=" * 80)
print("STEP 2: THE CLEANED UID FILE (shp_uid_final.csv)")
print("=" * 80)
print("""
This file takes the scrape results and:
  - Removes false positives (scraper matched wrong firm)
  - Removes schools, restaurants
  - Manual fixes for large employers
  - DEDUPLICATES TO ONE ROW PER UID
""")

print(f"  Total rows (= unique UIDs): {len(uid_final):,}")
print(f"  Unique mis_firmname values:  {uid_final['mis_firmname'].nunique():,}")
print(f"  Unique bur_firmname values:  {uid_final['bur_firmname'].dropna().nunique():,}")

# KEY: How many name variants were lost in deduplication?
scrape_clean = scrape[scrape['bur_uid'].notna()].copy()
names_in_scrape = scrape_clean.drop_duplicates(subset=['mis_firmname', 'bur_uid']).shape[0]
names_in_final = len(uid_final)

print(f"\n  >>> Name variants in scrape (with UID):  {names_in_scrape:,}")
print(f"  >>> Name variants in uid_final:           {names_in_final:,}")
print(f"  >>> LOST in deduplication:                 {names_in_scrape - names_in_final:,}")

# ============================================================


STEP 2: THE CLEANED UID FILE (shp_uid_final.csv)

This file takes the scrape results and:
  - Removes false positives (scraper matched wrong firm)
  - Removes schools, restaurants
  - Manual fixes for large employers
  - DEDUPLICATES TO ONE ROW PER UID

  Total rows (= unique UIDs): 7,014
  Unique mis_firmname values:  7,014
  Unique bur_firmname values:  7,012

  >>> Name variants in scrape (with UID):  10,446
  >>> Name variants in uid_final:           7,014
  >>> LOST in deduplication:                 3,432


In [6]:

# STEP 3: What is shp_x28_match.csv?
# ============================================================

print("\n" + "=" * 80)
print("STEP 3: THE X28 FIRM LIST (shp_x28_match.csv)")
print("=" * 80)
print("""
This file contains X28 firms that were matched to SHP via UID or name matching.
These are the firms for which we have job vacancy data.
""")

print(f"  Total X28 firms:            {len(x28):,}")
print(f"  With UID:                   {x28['uid'].notna().sum():,}")
print(f"  Without UID:                {x28['uid'].isna().sum():,}")



STEP 3: THE X28 FIRM LIST (shp_x28_match.csv)

This file contains X28 firms that were matched to SHP via UID or name matching.
These are the firms for which we have job vacancy data.

  Total X28 firms:            7,201
  With UID:                   6,907
  Without UID:                294


In [7]:
# ============================================================
# STEP 4: Where does uid_final connect to X28?
# ============================================================

print("\n" + "=" * 80)
print("STEP 4: CONNECTING UIDs TO X28 FIRMS")
print("=" * 80)

uid_final_uids = set(uid_final['uid'].dropna())
x28_uids = set(x28['uid'].dropna())
overlap = uid_final_uids & x28_uids

print(f"  UIDs in uid_final:           {len(uid_final_uids):,}")
print(f"  UIDs in X28:                 {len(x28_uids):,}")
print(f"  UIDs in BOTH:                {len(overlap):,}")
print(f"  In uid_final but NOT in X28: {len(uid_final_uids - x28_uids):,}")
print(f"  In X28 but NOT in uid_final: {len(x28_uids - uid_final_uids):,}")



STEP 4: CONNECTING UIDs TO X28 FIRMS
  UIDs in uid_final:           7,014
  UIDs in X28:                 6,898
  UIDs in BOTH:                4,837
  In uid_final but NOT in X28: 2,177
  In X28 but NOT in uid_final: 2,061


In [8]:
# ============================================================
# STEP 5: The person-level matching (0_anonymous_firmid.R)
# ============================================================

print("\n" + "=" * 80)
print("STEP 5: PERSON-LEVEL MATCHING (0_anonymous_firmid.R)")
print("=" * 80)
print("""
Script 3 builds lookup tables from:
  - X28 firm names
  - uid_final mis_firmname (ONE per UID)
  - uid_final bur_firmname (ONE per UID)
  - Original BUR/MIS PW52 names (direct match to X28 name)

Then for each person-year in BUR/MIS, tries to match their firm_name
against these lookups in 9 passes.
""")

print(f"  Person-year rows in firmid file:  {len(firmid):,}")
print(f"  Matched (have firm_id):           {firmid['firm_id'].notna().sum():,} ({100*firmid['firm_id'].notna().mean():.1f}%)")
print(f"  Unmatched (NA firm_id):           {firmid['firm_id'].isna().sum():,} ({100*firmid['firm_id'].isna().mean():.1f}%)")
print(f"  Unique persons matched:           {firmid[firmid['firm_id'].notna()]['idpers'].nunique():,}")
print(f"  Unique persons never matched:     {len(set(firmid['idpers']) - set(firmid[firmid['firm_id'].notna()]['idpers'])):,}")



STEP 5: PERSON-LEVEL MATCHING (0_anonymous_firmid.R)

Script 3 builds lookup tables from:
  - X28 firm names
  - uid_final mis_firmname (ONE per UID)
  - uid_final bur_firmname (ONE per UID)
  - Original BUR/MIS PW52 names (direct match to X28 name)

Then for each person-year in BUR/MIS, tries to match their firm_name
against these lookups in 9 passes.

  Person-year rows in firmid file:  70,111
  Matched (have firm_id):           29,912 (42.7%)
  Unmatched (NA firm_id):           40,199 (57.3%)
  Unique persons matched:           8,830
  Unique persons never matched:     8,239


In [9]:
# ============================================================
# STEP 6: THE KEY QUESTION — What's in the scrape file that
# could connect to X28 but is NOT being used?
# ============================================================

print("\n" + "=" * 80)
print("STEP 6: WHAT THE SCRAPE FILE KNOWS BUT ISN'T BEING USED")
print("=" * 80)
print("""
The scrape file has EVERY name variant with its UID.
The uid_final file keeps only ONE name per UID.
If we join scrape UIDs → X28 UIDs, we get a COMPLETE lookup
of all name variants → X28 firm_id.
""")

# Join scrape to X28 via UID
x28_uid_to_id = x28[x28['uid'].notna()][['uid', 'id', 'name']].drop_duplicates(subset='uid')
scrape_to_x28 = scrape[scrape['bur_uid'].notna()].merge(
    x28_uid_to_id, 
    left_on='bur_uid', 
    right_on='uid', 
    how='inner'
)

n_scrape_names_with_x28 = scrape_to_x28['mis_firmname'].nunique()
n_x28_firms_covered = scrape_to_x28['id'].nunique()

print(f"  Scrape name variants that map to an X28 firm via UID: {n_scrape_names_with_x28:,}")
print(f"  X28 firms these map to:                               {n_x28_firms_covered:,}")
print()

# Compare to what's currently in the lookup tables
# uid_final has 1 name per UID. How many of those UIDs are in X28?
uid_final_in_x28 = uid_final[uid_final['uid'].isin(x28_uids)]
print(f"  Currently used (uid_final names matching X28): {len(uid_final_in_x28):,} unique names")
print(f"  Could be used (scrape names matching X28):     {n_scrape_names_with_x28:,} unique names")
print(f"  ADDITIONAL names recoverable:                  {n_scrape_names_with_x28 - len(uid_final_in_x28):,}")



STEP 6: WHAT THE SCRAPE FILE KNOWS BUT ISN'T BEING USED

The scrape file has EVERY name variant with its UID.
The uid_final file keeps only ONE name per UID.
If we join scrape UIDs → X28 UIDs, we get a COMPLETE lookup
of all name variants → X28 firm_id.

  Scrape name variants that map to an X28 firm via UID: 6,346
  X28 firms these map to:                               4,667

  Currently used (uid_final names matching X28): 4,837 unique names
  Could be used (scrape names matching X28):     6,346 unique names
  ADDITIONAL names recoverable:                  1,509


In [10]:

# ============================================================
# STEP 7: CONCRETE UBS EXAMPLE
# ============================================================

print("\n" + "=" * 80)
print("STEP 7: CONCRETE EXAMPLE — UBS AG (UID CHE-101.329.561)")
print("=" * 80)

ubs_uid = 'CHE-101.329.561'

# All scrape entries for this UID
ubs_scrape = scrape[scrape['bur_uid'] == ubs_uid]
# uid_final entry
ubs_final = uid_final[uid_final['uid'] == ubs_uid]
# X28 entry
ubs_x28 = x28[x28['uid'] == ubs_uid]

print(f"\n  A) Scrape file: {len(ubs_scrape)} name variants all resolved to UID {ubs_uid}")
for _, r in ubs_scrape.iterrows():
    print(f"     \"{r['mis_firmname']}\" ({r['mis_firmloc']})")

print(f"\n  B) uid_final: ONLY {len(ubs_final)} entry kept after deduplication:")
for _, r in ubs_final.iterrows():
    print(f"     \"{r['mis_firmname']}\"")

print(f"\n  C) X28: firm name = \"{ubs_x28.iloc[0]['name']}\", id = {ubs_x28.iloc[0]['id']}")

print(f"\n  D) Script 3 lookup tables contain these UBS name variants:")
print(f"     - X28 name: \"{ubs_x28.iloc[0]['name']}\"")
print(f"     - uid_final mis_firmname: \"{ubs_final.iloc[0]['mis_firmname']}\"")
print(f"     - uid_final bur_firmname: \"{ubs_final.iloc[0]['bur_firmname']}\"")

in_lookup = {ubs_x28.iloc[0]['name'], ubs_final.iloc[0]['mis_firmname'], ubs_final.iloc[0]['bur_firmname']}
not_in_lookup = set(ubs_scrape['mis_firmname']) - in_lookup
print(f"\n  E) Names NOT in any lookup ({len(not_in_lookup)} variants):")
for n in sorted(not_in_lookup):
    print(f"     \"{n}\"  <-- person writes this, gets NO match")

print(f"\n  F) FIX: If we used the scrape file directly, ALL {len(ubs_scrape)} variants")
print(f"     would map to X28 id={ubs_x28.iloc[0]['id']} via their shared UID.")


STEP 7: CONCRETE EXAMPLE — UBS AG (UID CHE-101.329.561)

  A) Scrape file: 48 name variants all resolved to UID CHE-101.329.561
     "UBS AG Flurhof FA" (Zürich)
     "UBS AG" (Zürich)
     "UBS AG Flurhof FD" (Zürich)
     "UBS AG Flur-Nord Trakt A" (Zürich)
     "UBS AG WDR, Haus 2" (Glattbrugg)
     "UBS AG WDR, Haus 1" (Glattbrugg)
     "UBS AG" (Olten)
     "UBS SA" (Carouge GE)
     "UBS SA Centre des Acacias" (Carouge GE)
     "UBS AG" (Davos Platz)
     "UBS AG" (Horgen)
     "UBS SA" (Manno)
     "UBS AG" (Basel)
     "UBS AG VF Trakt A / Flur Süd" (Zürich)
     "UBS AG  Geschäftsleitung" (Basel)
     "UBS SA" (Renens VD)
     "UBS AG Dinocenter" (Zürich)
     "UBS AG" (Buchs SG)
     "UBS SA City Nuova" (Lugano)
     "UBS SA" (Chiasso)
     "UBS SA Palazzo Mercurio" (Chiasso)
     "UBS AG" (Zofingen)
     "UBS AG" (Aarau)
     "UBS AG Bahnhofstrassetrakt" (Zürich)
     "UBS AG Hochhaus zur Schanzenbrücke" (Zürich)
     "UBS AG" (Wädenswil)
     "UBS AG kappeli" (Zürich)
    

In [ ]:
ubs_final

,Unnamed: 0,mis_firmname,mis_firmloc,bur_firmname,bur_uid,bur_firmloc,alternative_uid,good,uid
44,45,UBS AG Flurhof FA,Zürich,UBS AG,CHE-101.329.561,Zürich,NaN,1,CHE-101.329.561


In [11]:
# ============================================================
# STEP 8: BUT WAIT — Does script 3 already handle this?
# ============================================================

print("\n" + "=" * 80)
print("STEP 8: DOES SCRIPT 3 ALREADY USE THE SCRAPE FILE?")
print("=" * 80)
print("""
Let me check what data sources script 3 actually loads:

From 0_anonymous_firmid.R:
  1. bur_in_shp <- read.csv("SHP_W13_to_W23_Bur_Prof-20230526.csv")  ← RAW BUR/MIS
  2. shp_uid_final <- read.csv("shp_uid_final.csv")                   ← DEDUPLICATED UIDs
  3. shp_x28_match <- read.csv("shp_x28_match.csv")                   ← X28 firms

It does NOT load scrape_uid_complete.csv!

The crosswalk is built from:
  - Source 1: uid_final mis_firmname joined to X28 via UID     (1 name per UID)
  - Source 2: uid_final mis_firmname joined to X28 via name    (exact match)
  - Source 3: uid_final bur_firmname joined to X28 via name    (exact match)
  - Source 4: BUR/MIS PW52 name joined to X28 via name         (exact match)

NONE of these use the full set of name→UID mappings from the scrape file.
""")

print("CONCLUSION:")
print("  The scrape file already proved that 'UBS AG Hauptsitz' = UID CHE-101.329.561 = UBS AG")
print("  But this information is LOST because uid_final only keeps 1 name per UID.")
print("  Script 3 never loads the scrape file to recover these mappings.")
print("  The manual fixes in script 3 (60+ UBS entries, 35+ Migros entries, etc.)")
print("  are literally re-doing work the scraper already did.")


STEP 8: DOES SCRIPT 3 ALREADY USE THE SCRAPE FILE?

Let me check what data sources script 3 actually loads:

From 0_anonymous_firmid.R:
  1. bur_in_shp <- read.csv("SHP_W13_to_W23_Bur_Prof-20230526.csv")  ← RAW BUR/MIS
  2. shp_uid_final <- read.csv("shp_uid_final.csv")                   ← DEDUPLICATED UIDs
  3. shp_x28_match <- read.csv("shp_x28_match.csv")                   ← X28 firms

It does NOT load scrape_uid_complete.csv!

The crosswalk is built from:
  - Source 1: uid_final mis_firmname joined to X28 via UID     (1 name per UID)
  - Source 2: uid_final mis_firmname joined to X28 via name    (exact match)
  - Source 3: uid_final bur_firmname joined to X28 via name    (exact match)
  - Source 4: BUR/MIS PW52 name joined to X28 via name         (exact match)

NONE of these use the full set of name→UID mappings from the scrape file.

CONCLUSION:
  The scrape file already proved that 'UBS AG Hauptsitz' = UID CHE-101.329.561 = UBS AG
  But this information is LOST because uid_final

In [12]:
# ============================================================
# STEP 9: QUANTIFY THE OPPORTUNITY
# ============================================================

print("\n" + "=" * 80)
print("STEP 9: HOW MANY ADDITIONAL MATCHES COULD WE GET?")
print("=" * 80)

# Build the complete lookup: scrape mis_firmname → X28 firm_id via UID
complete_lookup = scrape_to_x28[['mis_firmname', 'mis_firmloc', 'id', 'name']].copy()
complete_lookup = complete_lookup.rename(columns={'id': 'x28_id', 'name': 'x28_name'})
complete_lookup['firm_id'] = (complete_lookup['x28_id'] + 13) * 13

# Deduplicate: one firm_id per (mis_firmname, mis_firmloc)
complete_lookup = complete_lookup.drop_duplicates(subset=['mis_firmname', 'mis_firmloc'])

print(f"  Complete lookup (scrape → X28 via UID):")
print(f"    Unique (name, location) pairs: {len(complete_lookup):,}")
print(f"    Unique firm names:             {complete_lookup['mis_firmname'].nunique():,}")
print(f"    Mapping to X28 firms:          {complete_lookup['x28_id'].nunique():,}")
print()

# BUT: we don't know exactly how many person-years these cover because
# we don't have the raw BUR/MIS file with person IDs on this machine.
# The firmid file (shp_firmid_anon.csv) doesn't have firm names — just firm_id.

print("  LIMITATION: We cannot quantify exact person-year gains without the")
print("  raw BUR/MIS file, because shp_firmid_anon.csv has no firm names.")
print("  We need the unique (firm_name, location) list from the coauthor")
print("  to build the improved lookup, which he can then apply.")
print()

# What we CAN say: the scrape file has name→UID mappings for names
# that are NOT currently in any lookup table
current_lookup_names = set(uid_final['mis_firmname'].dropna()) | set(uid_final['bur_firmname'].dropna()) | set(x28['name'].dropna())
scrape_names_with_x28 = set(complete_lookup['mis_firmname'])
new_names = scrape_names_with_x28 - current_lookup_names

print(f"  Names currently in lookup tables: {len(current_lookup_names):,}")
print(f"  Names recoverable from scrape:    {len(scrape_names_with_x28):,}")
print(f"  NET NEW names (not in any current lookup): {len(new_names):,}")
print()
print(f"  These {len(new_names):,} firm name variants have KNOWN UIDs that map to X28 firms")
print(f"  but are currently not being used for person-level matching.")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"""
The scrape_uid_complete.csv file already contains the answer for many unmatched
persons. It has {len(scrape):,} (name, location) → UID mappings. Of these,
{len(complete_lookup):,} map to X28 firms via UID.

But shp_uid_final.csv deduplicates to 1 row per UID, keeping only {len(uid_final):,} names.
Script 3 builds its lookups from uid_final (not the scrape file), so it only
knows ~{len(current_lookup_names):,} name variants total.

{len(new_names):,} additional name variants could be recovered by using the scrape
file directly. These names already have verified UIDs linking them to X28 firms.

No fuzzy matching needed for these — just use the data that already exists.
Fuzzy matching would only be needed for the {n_no_uid:,} names where the scraper
found no UID at all.
""")



STEP 9: HOW MANY ADDITIONAL MATCHES COULD WE GET?
  Complete lookup (scrape → X28 via UID):
    Unique (name, location) pairs: 6,733
    Unique firm names:             6,346
    Mapping to X28 firms:          4,667

  LIMITATION: We cannot quantify exact person-year gains without the
  raw BUR/MIS file, because shp_firmid_anon.csv has no firm names.
  We need the unique (firm_name, location) list from the coauthor
  to build the improved lookup, which he can then apply.

  Names currently in lookup tables: 12,471
  Names recoverable from scrape:    6,346
  NET NEW names (not in any current lookup): 1,341

  These 1,341 firm name variants have KNOWN UIDs that map to X28 firms
  but are currently not being used for person-level matching.

SUMMARY

The scrape_uid_complete.csv file already contains the answer for many unmatched
persons. It has 13,643 (name, location) → UID mappings. Of these,
6,733 map to X28 firms via UID.

But shp_uid_final.csv deduplicates to 1 row per UID, keeping onl